# Exploration
In this notebook we explore the different existing datasets and models to perform ocr and/or recognition of "ancient Greek" text.

The goal of the notebook is to find the best model that perform line segmentation and text extraction one the datasets, may be a compilation of different source, in order to then apply these techniques on lambda users taken pictures.

## Assumptions
- Pictures will be tilted at maximum 20°
- Pictures will contain artifacts that are not text
- Pictures will be blurry, distorted, ...

## Datasets
As a first dataset we ll use a dataset recording historical greek handwritten document [https://zenodo.org/records/4095301](https://zenodo.org/records/4095301). It contains the image and the transcription, word by word and line by line. Trough the exploration we ll see if it is possible to augmente this dataset with geometric distortion, where blur or shade is for sure something that ll work.

In [ ]:
RUN_ON_COLAB = False

In [ ]:
from pathlib import Path

BASE_PATH = Path("EparchosData")

if RUN_ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_PATH = Path("drive/MyDrive/ASETEAM4")

In [ ]:
import re
import xml.etree.ElementTree as ET
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from matplotlib.patches import Rectangle
from PIL import Image
from scipy.ndimage import gaussian_filter, map_coordinates

DATA_DIR = BASE_PATH / Path("data")
DATA_DIR.mkdir(exist_ok=True)

ZENODO_RECORDS = {
    "eparchos": 4095301,
}

In [ ]:
def download_zenodo(name, record_id, out_dir=DATA_DIR, unzip=True):
    target = out_dir / name
    target.mkdir(parents=True, exist_ok=True)
    meta = requests.get(f"https://zenodo.org/api/records/{record_id}", timeout=30)
    meta.raise_for_status()
    for f in meta.json()["files"]:
        path = target / f["key"]
        if path.exists():
            print(f"skip {path} (already there)")
            continue
        print(f"downloading {f['key']} ({f['size'] / 1e6:.1f} MB)")
        with requests.get(f["links"]["self"], stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(path, "wb") as out:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    out.write(chunk)
        if unzip and path.suffix == ".zip":
            with zipfile.ZipFile(path) as z:
                z.extractall(target)
    return target

In [ ]:
for name, rid in ZENODO_RECORDS.items():
    download_zenodo(name, rid)

In [ ]:
EPARCHOS_IMAGES_DIR = DATA_DIR / Path("eparchos/eparchos/image")
eparchos_first_image = EPARCHOS_IMAGES_DIR / Path("6791 (1).png")

In [ ]:
img = Image.open(eparchos_first_image)
plt.imshow(img)

In [ ]:
EPARCHOS_XML_DIR = DATA_DIR / Path("eparchos/eparchos/xml")
eparchos_first_xml = EPARCHOS_XML_DIR / Path("6791 (1).xml")

In [ ]:
def parse_points(s):
    return np.array([[int(v) for v in p.split(",")] for p in s.split()])


def get_text(el):
    u = el.find("{*}TextEquiv/{*}Unicode")  # direct child only, not the words' text
    return u.text or "" if u is not None else ""


def bbox(poly):
    (x0, y0), (x1, y1) = poly.min(axis=0), poly.max(axis=0)
    return x0, y0, x1, y1


def parse_pagexml(xml_path):
    root = ET.parse(xml_path).getroot()  # ET handles the UTF-8 BOM
    page = root.find(".//{*}Page")
    meta = {
        "image": page.get("imageFilename"),
        "width": int(page.get("imageWidth")),
        "height": int(page.get("imageHeight")),
    }

    lines, words = [], []
    for region in page.findall("{*}TextRegion"):
        for line in region.findall("{*}TextLine"):
            poly = parse_points(line.find("{*}Coords").get("points"))
            raw = get_text(line)
            line_words = []
            for w in line.findall("{*}Word"):
                wpoly = parse_points(w.find("{*}Coords").get("points"))
                line_words.append(
                    {
                        "line_id": line.get("id"),
                        "word_id": w.get("id"),
                        "polygon": wpoly,
                        "bbox": bbox(wpoly),
                        "text": get_text(w),
                    }
                )
            line_words.sort(key=lambda d: d["bbox"][0])  # XML order != reading order
            words.extend(line_words)
            lines.append(
                {
                    "region_id": region.get("id"),
                    "line_id": line.get("id"),
                    "polygon": poly,
                    "bbox": bbox(poly),
                    "text_raw": raw,
                    "text": re.sub(r"\s+", " ", raw).strip(),  # collapse layout spaces
                    "text_from_words": " ".join(d["text"] for d in line_words),
                    "n_words": len(line_words),
                }
            )
    return pd.DataFrame(lines), pd.DataFrame(words), meta


lines_df, words_df, meta = parse_pagexml(eparchos_first_xml)
print(meta)
lines_df[["line_id", "bbox", "n_words", "text"]]

In [ ]:
img = Image.open(EPARCHOS_IMAGES_DIR / meta["image"])  # filename taken from the XML
assert img.size == (meta["width"], meta["height"]), (img.size, meta)

fig, ax = plt.subplots()
ax.imshow(img, cmap="gray")
for df, color, lw in [(lines_df, "red", 1.2), (words_df, "lime", 0.6)]:
    for x0, y0, x1, y1 in df["bbox"]:
        ax.add_patch(
            Rectangle(
                (x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, linewidth=lw
            )
        )
for i, (x0, y0, _, _) in enumerate(lines_df["bbox"]):
    ax.text(x0 - 4, y0 + 15, str(i), color="blue", fontsize=8, ha="right")

In [ ]:
plt.rcParams["font.family"] = "DejaVu Sans"  # renders polytonic Greek

n = len(lines_df)
fig, axes = plt.subplots(n, 1, figsize=(12, 1.2 * n))
for ax, (_, row) in zip(axes, lines_df.iterrows(), strict=False):
    ax.imshow(img.crop(row["bbox"]), cmap="gray")
    ax.set_title(f"{row['line_id']}: {row['text']}", fontsize=10, loc="left")
    ax.axis("off")
plt.tight_layout()

## Applying Augmentation for Both Images and Labels
### Distortion exploration

In [ ]:
# --- Polygon helpers -----------------------------------------------------------


def densify(poly, step=4.0):
    """Add points along each edge so a rectangle can bend under non-linear warps."""
    poly = np.asarray(poly, dtype=np.float32)
    out = []
    for a, b in zip(poly, np.roll(poly, -1, axis=0), strict=False):
        n = max(int(np.ceil(np.linalg.norm(b - a) / step)), 1)
        t = np.linspace(0, 1, n, endpoint=False)[:, None]
        out.append(a + t * (b - a))
    return np.concatenate(out)


def polygon_bbox(poly):
    x0, y0 = np.floor(poly.min(axis=0)).astype(int)
    x1, y1 = np.ceil(poly.max(axis=0)).astype(int)
    return int(x0), int(y0), int(x1), int(y1)


# --- Geometric transforms (image + polygons) ----------------------------------


def perspective(
    img, polys, rng, max_corner_shift=0.08, max_rotation=6, background=None
):
    """Camera not parallel to the page: random corner shift + small rotation.
    The output canvas grows so the whole page stays visible."""
    h, w = img.shape[:2]
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = src + rng.uniform(-max_corner_shift, max_corner_shift, (4, 2)).astype(
        np.float32
    ) * [w, h]

    angle = np.deg2rad(rng.uniform(-max_rotation, max_rotation))
    c = np.float32([w / 2, h / 2])
    R = np.float32([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    dst = (dst - c) @ R.T + c

    H = cv2.getPerspectiveTransform(src, dst.astype(np.float32))

    corners = cv2.perspectiveTransform(src[None], H)[0]
    margin = int(0.03 * max(w, h))
    (xmin, ymin), (xmax, ymax) = corners.min(axis=0), corners.max(axis=0)
    T = np.array(
        [[1, 0, margin - xmin], [0, 1, margin - ymin], [0, 0, 1]], dtype=np.float64
    )
    H = T @ H
    out_size = (
        int(np.ceil(xmax - xmin)) + 2 * margin,
        int(np.ceil(ymax - ymin)) + 2 * margin,
    )

    if background is None:
        background = int(rng.integers(20, 90))  # dark table
    border = background if img.ndim == 2 else (background,) * img.shape[2]
    out = cv2.warpPerspective(
        img,
        H,
        out_size,
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=border,
    )

    new_polys = [
        cv2.perspectiveTransform(p.reshape(1, -1, 2).astype(np.float32), H)[0]
        for p in polys
    ]
    return out, new_polys


def _apply_displacement(img, polys, dx, dy, n_iter=30):
    """Image: output (x, y) reads input (x + dx, y + dy) via cv2.remap.
    Polygons: inverse mapping found by fixed-point iteration q = p - d(q)."""
    h, w = img.shape[:2]
    xs, ys = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    out = cv2.remap(
        img,
        xs + dx.astype(np.float32),
        ys + dy.astype(np.float32),
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REPLICATE,
    )

    def sample(field, q):
        return map_coordinates(field, [q[:, 1], q[:, 0]], order=1, mode="nearest")

    new_polys = []
    for p in polys:
        p = np.asarray(p, dtype=np.float64)
        q = p.copy()
        for _ in range(n_iter):
            q = p - np.stack([sample(dx, q), sample(dy, q)], axis=1)
        new_polys.append(q.astype(np.float32))
    return out, new_polys


def page_curl(img, polys, rng, max_amplitude=0.012, period_range=(0.7, 1.6)):
    """Paper not flat: lines become waves."""
    h, w = img.shape[:2]
    xs, ys = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    a_y = rng.uniform(0.3, 1.0) * max_amplitude * h
    a_x = rng.uniform(0.0, 0.5) * max_amplitude * w
    lam_x = rng.uniform(*period_range) * w
    lam_y = rng.uniform(*period_range) * h
    dy = a_y * np.sin(2 * np.pi * xs / lam_x + rng.uniform(0, 2 * np.pi))
    dx = a_x * np.sin(2 * np.pi * ys / lam_y + rng.uniform(0, 2 * np.pi))
    return _apply_displacement(img, polys, dx, dy)


def elastic(img, polys, rng, amplitude=0.006, smoothness=0.08):
    """Random smooth deformation, scaled to the page size."""
    h, w = img.shape[:2]
    sigma = smoothness * max(h, w)
    fields = []
    for _ in range(2):
        f = gaussian_filter(rng.uniform(-1, 1, (h, w)), sigma)
        fields.append(f / (np.abs(f).max() + 1e-8) * amplitude * max(h, w))
    return _apply_displacement(img, polys, *fields)


# --- Photometric transforms (image only) --------------------------------------


def uneven_lighting(img, rng, strength=0.45):
    h, w = img.shape[:2]
    xs, ys = np.meshgrid(np.linspace(-1, 1, w), np.linspace(-1, 1, h))
    theta = rng.uniform(0, 2 * np.pi)
    gradient = (
        (np.cos(theta) * xs + np.sin(theta) * ys) * rng.uniform(0.3, 1.0) * strength / 2
    )
    cx, cy = rng.uniform(-1, 1, 2)
    shadow = (
        -rng.uniform(0.2, 0.6)
        * strength
        * np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / rng.uniform(0.1, 0.5))
    )
    factor = 1 + gradient + shadow
    if img.ndim == 3:
        factor = factor[..., None]
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)


def blur(img, rng, max_sigma=1.5):
    if rng.random() < 0.5:  # defocus
        return cv2.GaussianBlur(img, (0, 0), rng.uniform(0.3, max_sigma))
    k = int(rng.integers(3, 8))  # motion blur
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2] = 1.0 / k
    M = cv2.getRotationMatrix2D(((k - 1) / 2, (k - 1) / 2), rng.uniform(0, 180), 1.0)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    return cv2.filter2D(img, -1, kernel / kernel.sum())


def sensor_noise(img, rng, max_std=10):
    noise = rng.normal(0, rng.uniform(2, max_std), img.shape)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def jpeg(img, rng, quality_range=(30, 80)):
    ok, buf = cv2.imencode(
        ".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(*quality_range))]
    )
    return cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)


# --- Full pipeline ------------------------------------------------------------


def distort_page(
    img, polys, rng, p_curl=0.7, p_elastic=0.5, photometric=True, step=4.0
):
    """img: HxW or HxWxC uint8 array. polys: list of (N, 2) arrays.
    Returns the distorted image and the matching (densified) polygons."""
    polys = [densify(p, step) for p in polys]
    if rng.random() < p_curl:
        img, polys = page_curl(img, polys, rng)
    if rng.random() < p_elastic:
        img, polys = elastic(img, polys, rng)
    img, polys = perspective(img, polys, rng)
    if photometric:
        img = uneven_lighting(img, rng)
        img = blur(img, rng)
        img = sensor_noise(img, rng)
        img = jpeg(img, rng)
    return img, polys


# --- Check that labels still fit ----------------------------------------------


def label_alignment_iou(img_shape, polys, rng, **kwargs):
    """IoU between each polygon distorted as a mask image and the same polygon
    distorted as points. Close to 1 = labels still fit."""
    h, w = img_shape[:2]
    seed = int(rng.integers(1 << 31))
    ious = []
    for i, p in enumerate(polys):
        mask = np.zeros((h, w), np.uint8)
        cv2.fillPoly(mask, [np.round(p).astype(np.int32)], 255)
        m, new_polys = distort_page(
            mask, polys, np.random.default_rng(seed), photometric=False, **kwargs
        )
        drawn = np.zeros(m.shape, np.uint8)
        cv2.fillPoly(drawn, [np.round(new_polys[i]).astype(np.int32)], 1)
        warped = m > 127
        inter = np.logical_and(warped, drawn).sum()
        union = np.logical_or(warped, drawn).sum()
        ious.append(inter / union if union else 1.0)
    return np.array(ious)

In [ ]:
img_np = np.array(img)  # PIL image from before
polys = list(lines_df["polygon"])

rng = np.random.default_rng(42)
img_d, polys_d = distort_page(img_np, polys, rng)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
for ax, im, ps, title in [
    (axes[0], img_np, polys, "original"),
    (axes[1], img_d, polys_d, "distorted"),
]:
    ax.imshow(im, cmap="gray")
    for p in ps:
        closed = np.vstack([p, p[:1]])
        ax.plot(closed[:, 0], closed[:, 1], "r-", linewidth=0.8)
    ax.set_title(title)
    ax.axis("off")
plt.show()

print("alignment IoU per line:", label_alignment_iou(img_np.shape, polys, rng).round(3))

In [ ]:
distorted_df = lines_df.copy()
distorted_df["polygon"] = polys_d
distorted_df["bbox"] = [polygon_bbox(p) for p in polys_d]

fig, axes = plt.subplots(6, 1, figsize=(12, 8))
for ax, (_, row) in zip(axes, distorted_df.head(6).iterrows(), strict=False):
    x0, y0, x1, y1 = row["bbox"]
    ax.imshow(img_d[max(y0, 0) : y1, max(x0, 0) : x1], cmap="gray")
    ax.set_title(row["text"], fontsize=10, loc="left")
    ax.axis("off")
plt.tight_layout()

In [ ]:
# Every augmentation has the same signature:
#     aug(img, polys, rng, **params) -> (img, polys)
# img: HxW or HxWxC uint8 array, polys: list of (N, 2) float arrays (densified).
# Photometric ones return polys unchanged; geometric ones move them.


# --- Helpers ------------------------------------------------------------------
def polygon_bbox(poly):
    x0, y0 = np.floor(poly.min(axis=0)).astype(int)
    x1, y1 = np.ceil(poly.max(axis=0)).astype(int)
    return int(x0), int(y0), int(x1), int(y1)


def _border(img, value):
    return value if img.ndim == 2 else (value,) * img.shape[2]


def _warp_homography(img, polys, H, background):
    """Apply a 3x3 homography
    to image and polygons, growing the canvas so nothing is cut."""
    h, w = img.shape[:2]
    corners = cv2.perspectiveTransform(
        np.float32([[[0, 0], [w, 0], [w, h], [0, h]]]), H
    )[0]
    margin = int(0.03 * max(w, h))
    (xmin, ymin), (xmax, ymax) = corners.min(axis=0), corners.max(axis=0)
    T = np.array(
        [[1, 0, margin - xmin], [0, 1, margin - ymin], [0, 0, 1]], dtype=np.float64
    )
    H = T @ H
    size = (
        int(np.ceil(xmax - xmin)) + 2 * margin,
        int(np.ceil(ymax - ymin)) + 2 * margin,
    )
    out = cv2.warpPerspective(
        img,
        H,
        size,
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=_border(img, background),
    )
    new_polys = [
        cv2.perspectiveTransform(p.reshape(1, -1, 2).astype(np.float32), H)[0]
        for p in polys
    ]
    return out, new_polys


def _warp_displacement(img, polys, dx, dy, n_iter=30):
    """Output pixel (x, y) reads input (x + dx, y + dy). Points use the inverse
    mapping, found by fixed-point iteration (valid for smooth fields)."""
    h, w = img.shape[:2]
    xs, ys = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    out = cv2.remap(
        img,
        xs + dx.astype(np.float32),
        ys + dy.astype(np.float32),
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REPLICATE,
    )

    def sample(field, q):
        return map_coordinates(field, [q[:, 1], q[:, 0]], order=1, mode="nearest")

    new_polys = []
    for p in polys:
        p = np.asarray(p, dtype=np.float64)
        q = p.copy()
        for _ in range(n_iter):
            q = p - np.stack([sample(dx, q), sample(dy, q)], axis=1)
        new_polys.append(q.astype(np.float32))
    return out, new_polys


def _scale_light(img, factor):
    if img.ndim == 3:
        factor = factor[..., None]
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)


# --- Geometric augmentations (image + polygons) -------------------------------


def tilt(img, polys, rng, max_angle=10, background=None):
    """In-plane rotation: phone held at an angle."""
    h, w = img.shape[:2]
    angle = rng.uniform(-max_angle, max_angle)
    H = np.vstack([cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0), [0, 0, 1]])
    bg = int(rng.integers(20, 90)) if background is None else background
    return _warp_homography(img, polys, H, bg)


def scale_shift(
    img, polys, rng, scale_range=(0.75, 1.15), max_shift=0.05, background=None
):
    """Page closer or further from the camera, not centered."""
    h, w = img.shape[:2]
    s = rng.uniform(*scale_range)
    tx, ty = rng.uniform(-max_shift, max_shift, 2) * [w, h]
    H = np.array([[s, 0, tx], [0, s, ty], [0, 0, 1]], dtype=np.float64)
    bg = int(rng.integers(20, 90)) if background is None else background
    return _warp_homography(img, polys, H, bg)


def book_fold(img, polys, rng, max_depth=0.04):
    """Page near a book spine: one side bends away, lines curve near the edge."""
    h, w = img.shape[:2]
    xs, ys = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    side = rng.choice([-1, 1])  # spine on the left or right
    u = xs / w if side == 1 else 1 - xs / w  # 0 at the spine
    depth = rng.uniform(0.5, 1.0) * max_depth * h
    bend = depth * (1 - u) ** 3  # strong near the spine, flat elsewhere
    dy = bend * (ys / h - 0.5) * 2  # pinch towards the vertical middle
    dx = side * bend * 0.5
    return _warp_displacement(img, polys, dx, dy)


def distortion(img, polys, rng, amplitude=0.008, smoothness=0.07):
    """Random smooth elastic deformation (crumpled paper)."""
    h, w = img.shape[:2]
    sigma = smoothness * max(h, w)
    fields = []
    for _ in range(2):
        f = gaussian_filter(rng.uniform(-1, 1, (h, w)), sigma)
        fields.append(f / (np.abs(f).max() + 1e-8) * amplitude * max(h, w))
    return _warp_displacement(img, polys, *fields)


# --- Photometric augmentations (image only) -----------------------------------
def uneven_lighting_polys(img, polys, rng, strength=0.45):
    h, w = img.shape[:2]
    xs, ys = np.meshgrid(np.linspace(-1, 1, w), np.linspace(-1, 1, h))
    theta = rng.uniform(0, 2 * np.pi)
    gradient = (
        (np.cos(theta) * xs + np.sin(theta) * ys) * rng.uniform(0.3, 1.0) * strength / 2
    )
    cx, cy = rng.uniform(-1, 1, 2)
    shadow = (
        -rng.uniform(0.2, 0.6)
        * strength
        * np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / rng.uniform(0.1, 0.5))
    )
    return _scale_light(img, 1 + gradient + shadow), polys


def blur_polys(img, polys, rng, max_sigma=1.5):
    """Defocus blur (hand-shake blur is the separate `motion_blur`)."""
    return cv2.GaussianBlur(img, (0, 0), rng.uniform(0.3, max_sigma)), polys


def jpeg_polys(img, polys, rng, quality_range=(30, 80)):
    ok, buf = cv2.imencode(
        ".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(*quality_range))]
    )
    return cv2.imdecode(buf, cv2.IMREAD_UNCHANGED), polys


def motion_blur(img, polys, rng, length_range=(5, 13)):
    """Hand shake during the shot."""
    k = int(rng.integers(*length_range)) | 1
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2] = 1.0
    M = cv2.getRotationMatrix2D((k // 2, k // 2), rng.uniform(0, 180), 1.0)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    return cv2.filter2D(img, -1, kernel / kernel.sum()), polys


def shade(img, polys, rng, darkness_range=(0.35, 0.65), softness_range=(0.02, 0.12)):
    """Shadow of the phone or hand: one side of a random line is darker, soft edge."""
    h, w = img.shape[:2]
    xs, ys = np.meshgrid(np.linspace(-1, 1, w), np.linspace(-1, 1, h))
    theta = rng.uniform(0, 2 * np.pi)
    offset = rng.uniform(-0.5, 0.5)
    dist = np.cos(theta) * xs + np.sin(theta) * ys - offset
    mask = 1 / (
        1 + np.exp(-dist / rng.uniform(*softness_range))
    )  # 0 -> 1 across the edge
    darkness = rng.uniform(*darkness_range)
    return _scale_light(img, 1 - darkness * mask), polys


def glare(img, polys, rng, intensity_range=(0.5, 0.9), size_range=(0.1, 0.3)):
    """Flash / lamp reflection: a bright spot that washes out the ink."""
    h, w = img.shape[:2]
    xs, ys = np.meshgrid(np.linspace(-1, 1, w), np.linspace(-1, 1, h))
    cx, cy = rng.uniform(-0.7, 0.7, 2)
    sx, sy = rng.uniform(*size_range, 2)
    spot = np.exp(-(((xs - cx) / sx) ** 2 + ((ys - cy) / sy) ** 2))
    alpha = rng.uniform(*intensity_range) * spot
    if img.ndim == 3:
        alpha = alpha[..., None]
    out = img.astype(np.float32) * (1 - alpha) + 255 * alpha
    return np.clip(out, 0, 255).astype(np.uint8), polys


def contrast_brightness(
    img, polys, rng, contrast_range=(0.5, 1.3), brightness_range=(-40, 40)
):
    """Under/over-exposed photo, faded ink."""
    c = rng.uniform(*contrast_range)
    b = rng.uniform(*brightness_range)
    out = (img.astype(np.float32) - 128) * c + 128 + b
    return np.clip(out, 0, 255).astype(np.uint8), polys


def noise(img, polys, rng, std_range=(8, 20)):
    """Sensor noise in low light."""
    out = img.astype(np.float32) + rng.normal(0, rng.uniform(*std_range), img.shape)
    return np.clip(out, 0, 255).astype(np.uint8), polys


def low_resolution(img, polys, rng, factor_range=(0.3, 0.6)):
    """Photo taken from far away then zoomed: lost detail, same size and labels."""
    h, w = img.shape[:2]
    f = rng.uniform(*factor_range)
    small = cv2.resize(
        img, (max(1, int(w * f)), max(1, int(h * f))), interpolation=cv2.INTER_AREA
    )
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR), polys


# --- Registry and random pipeline ---------------------------------------------

AUGMENTATIONS = {
    "blur": blur_polys,
    "motion blur": motion_blur,
    "shade": shade,
    "uneven lighting": uneven_lighting_polys,
    "glare": glare,
    "contrast / brightness": contrast_brightness,
    "noise": noise,
    "low resolution": low_resolution,
    "jpeg": jpeg_polys,
    "tilt": tilt,
    "perspective": perspective,
    "scale / shift": scale_shift,
    "distortion": distortion,
    "page curl": page_curl,
    "book fold": book_fold,
}

CANVAS_CHANGING = {"scale / shift", "tilt", "perspective"}
GEOMETRIC = [
    "page curl",
    "book fold",
    "distortion",
    "scale / shift",
    "tilt",
    "perspective",
]
PHOTOMETRIC = [
    "shade",
    "uneven lighting",
    "glare",
    "contrast / brightness",
    "blur",
    "motion blur",
    "low resolution",
    "noise",
    "jpeg",
]


def random_augment(img, polys, rng, p=0.4, step=4.0):
    """Apply each augmentation with probability p. Geometric first (the canvas-changing
    ones last), then photometric, in the order a real photo would degrade."""
    polys = [densify(p_, step) for p_ in polys]
    background = int(
        rng.integers(20, 90)
    )  # same "table" colour for every canvas change
    for name in GEOMETRIC + PHOTOMETRIC:
        if rng.random() < p:
            kwargs = {"background": background} if name in CANVAS_CHANGING else {}
            img, polys = AUGMENTATIONS[name](img, polys, rng, **kwargs)
    return img, polys


# --- Grid plot ----------------------------------------------------------------


def _draw(ax, img, polys, title, show_labels):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    if show_labels:
        for p in polys:
            closed = np.vstack([p, p[:1]])
            ax.plot(closed[:, 0], closed[:, 1], "r-", linewidth=0.6)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


def plot_augmentation_grid(
    img, polys, seed=0, n_cols=4, n_random=1, show_labels=True, cell_size=4
):
    """One cell per augmentation applied alone, plus n_random full random pipelines."""
    img = np.asarray(img)
    base = [densify(p) for p in polys]
    panels = [("original", img, base)]
    for i, (name, fn) in enumerate(AUGMENTATIONS.items()):
        out, ps = fn(img, base, np.random.default_rng(seed + i))
        panels.append((name, out, ps))
    for j in range(n_random):
        out, ps = random_augment(img, polys, np.random.default_rng(seed + 1000 + j))
        panels.append((f"random pipeline #{j + 1}", out, ps))

    n_rows = int(np.ceil(len(panels) / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(cell_size * n_cols, cell_size * n_rows)
    )
    for ax in axes.flat:
        ax.axis("off")
    for ax, (title, im, ps) in zip(axes.flat, panels, strict=False):
        _draw(ax, im, ps, title, show_labels)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_augmentation_grid(np.array(img), list(lines_df["polygon"]), seed=0, n_random=5)

# Augmentation

In [ ]:
# --- Augmented dataset: levels, augmentation, ground truth for staged evaluation ---
import hashlib
import io
import json
import unicodedata
import zlib

try:
    from tqdm.auto import tqdm
except ImportError:  # tqdm is optional

    def tqdm(it, **kwargs):
        return it


EPARCHOS_ROOT = DATA_DIR / "eparchos"  # original dataset
AUG_DIR = DATA_DIR / "eparchos_aug"  # augmented dataset, next to it
AUG_SEED = 2026  # global seed -> reproducible dataset
PIPELINE_VERSION = 2  # bump when the build logic changes

# Each level: augmentation name -> (probability, kwargs overriding the defaults).
# `min_ops`: at least that many augmentations per page.
# `occlusion`: with probability p, erase n_words words (ink removed, flagged in the
# ground truth) to test whether a model invents text it cannot see.
# "original" applies nothing: same files and annotations as the levels, as reference.
LEVELS = {
    "original": {"min_ops": 0, "ops": {}, "occlusion": None},
    "light": {
        "min_ops": 2,
        "occlusion": None,
        "ops": {
            "scale / shift": (0.3, {"scale_range": (0.95, 1.05), "max_shift": 0.02}),
            "tilt": (0.5, {"max_angle": 3}),
            "perspective": (0.3, {"max_corner_shift": 0.02, "max_rotation": 2}),
            "uneven lighting": (0.4, {"strength": 0.15}),
            "contrast / brightness": (
                0.5,
                {"contrast_range": (0.85, 1.1), "brightness_range": (-15, 15)},
            ),
            "blur": (0.3, {"max_sigma": 0.7}),
            "noise": (0.4, {"std_range": (2, 5)}),
            "jpeg": (0.5, {"quality_range": (70, 95)}),
        },
    },
    "moderate": {
        "min_ops": 4,
        "occlusion": {"p": 0.5, "n_words": (1, 2)},
        "ops": {
            "page curl": (0.4, {"max_amplitude": 0.007}),
            "distortion": (0.3, {"amplitude": 0.004}),
            "scale / shift": (0.4, {"scale_range": (0.85, 1.1), "max_shift": 0.04}),
            "tilt": (0.6, {"max_angle": 8}),
            "perspective": (0.6, {"max_corner_shift": 0.05, "max_rotation": 4}),
            "shade": (0.4, {"darkness_range": (0.2, 0.45)}),
            "uneven lighting": (0.6, {"strength": 0.35}),
            "glare": (0.3, {"intensity_range": (0.3, 0.6)}),
            "contrast / brightness": (
                0.6,
                {"contrast_range": (0.65, 1.2), "brightness_range": (-30, 30)},
            ),
            "blur": (0.5, {"max_sigma": 1.2}),
            "motion blur": (0.3, {"length_range": (3, 8)}),
            "low resolution": (0.3, {"factor_range": (0.5, 0.75)}),
            "noise": (0.6, {"std_range": (5, 12)}),
            "jpeg": (0.7, {"quality_range": (45, 80)}),
        },
    },
    "heavy": {
        "min_ops": 7,
        "occlusion": {"p": 0.7, "n_words": (1, 3)},
        "ops": {
            "page curl": (0.7, {"max_amplitude": 0.012}),
            "book fold": (0.5, {"max_depth": 0.04}),
            "distortion": (0.5, {"amplitude": 0.008}),
            "scale / shift": (0.6, {"scale_range": (0.7, 1.15), "max_shift": 0.06}),
            "tilt": (0.9, {"max_angle": 20}),  # matches the "max 20°" assumption
            "perspective": (0.9, {"max_corner_shift": 0.08, "max_rotation": 6}),
            "shade": (0.6, {"darkness_range": (0.35, 0.65)}),
            "uneven lighting": (0.8, {"strength": 0.5}),
            "glare": (0.5, {"intensity_range": (0.5, 0.85)}),
            "contrast / brightness": (
                0.8,
                {"contrast_range": (0.45, 1.3), "brightness_range": (-45, 45)},
            ),
            "blur": (0.6, {"max_sigma": 1.8}),
            "motion blur": (0.5, {"length_range": (7, 14)}),
            "low resolution": (0.5, {"factor_range": (0.3, 0.5)}),
            "noise": (0.8, {"std_range": (10, 22)}),
            "jpeg": (0.9, {"quality_range": (25, 55)}),
        },
    },
}


def level_fingerprint(level):
    """Short hash of a level's config + seed + build version: detects stale builds."""
    blob = json.dumps(
        {"seed": AUG_SEED, "v": PIPELINE_VERSION, "cfg": LEVELS[level]},
        sort_keys=True,
        default=str,
    )
    return hashlib.sha1(blob.encode()).hexdigest()[:12]


def page_rng(level, stem):
    """One deterministic RNG per (level, page): demo == dataset, resumable builds."""
    return np.random.default_rng(
        [AUG_SEED, zlib.crc32(level.encode()), zlib.crc32(stem.encode())]
    )


# --- Text ----------------------------------------------------------------------


def normalize_text(s):
    """NFC (same as the predictors) + collapsed whitespace."""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", s or "")).strip()


def strip_punct(s):
    return "".join(
        c for c in normalize_text(s) if not unicodedata.category(c).startswith("P")
    )


# --- Geometry helpers ----------------------------------------------------------


def densify_open(line, step=4.0):
    """densify() for polylines (baselines): no closing edge."""
    line = np.asarray(line, dtype=np.float32)
    out = []
    for a, b in zip(line[:-1], line[1:], strict=False):
        n = max(int(np.ceil(np.linalg.norm(b - a) / step)), 1)
        t = np.linspace(0, 1, n, endpoint=False)[:, None]
        out.append(a + t * (b - a))
    out.append(line[-1:])
    return np.concatenate(out)


def finalize_shape(p, w, h, closed):
    """Clip to the image, round to ints, drop the redundant densified points."""
    p = np.round(np.clip(p, [0, 0], [w - 1, h - 1])).astype(np.int32)
    keep = np.r_[True, np.any(np.diff(p, axis=0) != 0, axis=1)]
    p = p[keep]
    if len(p) >= 3:
        p = cv2.approxPolyDP(p.reshape(-1, 1, 2), 0.75, closed).reshape(-1, 2)
    return p


def box_union(boxes):
    boxes = [b for b in boxes if b is not None]
    if not boxes:
        return None
    b = np.array(boxes)
    return (
        int(b[:, 0].min()),
        int(b[:, 1].min()),
        int(b[:, 2].max()),
        int(b[:, 3].max()),
    )


def to_gray(img):
    return img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)


def load_page_image(path):
    im = Image.open(path)
    if im.mode not in ("L", "RGB"):
        im = im.convert("RGB")
    return np.array(im)


# --- Occlusion and ink boxes ----------------------------------------------------


def occlude_words(img, word_polys, word_texts, rng, cfg, min_len=3):
    """Erase whole words (paint them with the surrounding paper colour).
    Only words with >= min_len letters,
    so 'reading' them from context is a real test."""
    candidates = [i for i, t in enumerate(word_texts) if len(strip_punct(t)) >= min_len]
    if not cfg or not candidates or rng.random() >= cfg["p"]:
        return img, []
    lo, hi = cfg["n_words"]
    n = min(int(rng.integers(lo, hi + 1)), len(candidates))
    chosen = sorted(int(i) for i in rng.choice(candidates, size=n, replace=False))
    out, gray = img.copy(), to_gray(img)
    for i in chosen:
        mask = np.zeros(img.shape[:2], np.uint8)
        cv2.fillPoly(mask, [np.round(word_polys[i]).astype(np.int32)], 1)
        mask = cv2.dilate(mask, np.ones((3, 3), np.uint8)).astype(bool)
        ring = (
            cv2.dilate(mask.astype(np.uint8), np.ones((15, 15), np.uint8)).astype(bool)
            & ~mask
        )
        paper = gray[ring] >= np.median(gray[ring])  # bright = paper, not ink
        out[mask] = np.median(img[ring][paper], axis=0)
    return out, chosen


def ink_boxes(img, polys, min_area=6):
    """Tight box around the ink inside each polygon (None if no ink).
    Otsu threshold computed on the pixels of all polygons together (paper vs ink)."""
    gray = to_gray(img)
    h, w = gray.shape
    union = np.zeros((h, w), np.uint8)
    cv2.fillPoly(union, [np.round(p).astype(np.int32) for p in polys], 1)
    vals = gray[union.astype(bool)]
    if vals.size == 0:
        return [None] * len(polys)
    thr, _ = cv2.threshold(
        vals.reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    out = []
    for p in polys:
        x0, y0, x1, y1 = polygon_bbox(np.asarray(p, np.float32))
        x0, y0, x1, y1 = max(x0, 0), max(y0, 0), min(x1 + 1, w), min(y1 + 1, h)
        m = np.zeros((y1 - y0, x1 - x0), np.uint8)
        cv2.fillPoly(m, [np.round(np.asarray(p) - [x0, y0]).astype(np.int32)], 1)
        ink = ((gray[y0:y1, x0:x1] <= thr) & m.astype(bool)).astype(np.uint8)
        n, _, stats, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
        keep = [s for s in stats[1:] if s[cv2.CC_STAT_AREA] >= min_area]  # drop specks
        if not keep:
            out.append(None)
            continue
        s = np.array(keep)
        out.append(
            (
                int(x0 + s[:, 0].min()),
                int(y0 + s[:, 1].min()),
                int(x0 + (s[:, 0] + s[:, 2]).max() - 1),
                int(y0 + (s[:, 1] + s[:, 3]).max() - 1),
            )
        )
    return out


# --- Augmentation --------------------------------------------------------------


def choose_ops(rng, cfg):
    names = [n for n in GEOMETRIC + PHOTOMETRIC if n in cfg["ops"]]  # realistic order
    fire = {n: rng.random() < cfg["ops"][n][0] for n in names}
    missing = cfg["min_ops"] - sum(fire.values())
    if missing > 0:  # top up to min_ops with augmentations that didn't fire
        off = [n for n in names if not fire[n]]
        for n in rng.choice(off, size=min(missing, len(off)), replace=False):
            fire[n] = True
    return [n for n in names if fire[n]]


def apply_ops(img, pts, rng, names, cfg, background):
    for name in names:
        kwargs = dict(cfg["ops"][name][1])
        if name in CANVAS_CHANGING:
            kwargs["background"] = background
        img, pts = AUGMENTATIONS[name](img, pts, rng, **kwargs)
    return img, pts


def augment_pagexml(xml_path, level, image_dir=None, step=4.0):
    """Original page -> augmented image + updated PAGE-XML + full ground truth.

    Order: occlusion (damage on the page) -> geometric ops (image AND every
    polygon of the XML in one pass) -> ink boxes measured on this clean warped
    image -> photometric ops (image only)."""
    xml_path = Path(xml_path)
    image_dir = Path(image_dir or xml_path.parent.parent / "image")
    cfg = LEVELS[level]
    tree = ET.parse(xml_path)
    page = tree.getroot().find(".//{*}Page")
    img = load_page_image(image_dir / page.get("imageFilename"))

    # every element carrying geometry: Coords of regions/lines/words + Baselines
    elements = [el for el in page.iter() if (el.get("points") or "").strip()]
    position = {id(el): k for k, el in enumerate(elements)}
    shapes = [
        (parse_points(el.get("points")), not el.tag.endswith("Baseline"))
        for el in elements
    ]
    word_els = [
        w
        for w in page.iter()
        if w.tag.endswith("}Word") and w.find("{*}Coords") is not None
    ]
    word_ids = [w.get("id") for w in word_els]
    word_k = [position[id(w.find("{*}Coords"))] for w in word_els]

    rng = page_rng(level, xml_path.stem)
    ops = choose_ops(rng, cfg)
    img, occluded = occlude_words(
        img,
        [shapes[k][0] for k in word_k],
        [get_text(w) for w in word_els],
        rng,
        cfg["occlusion"],
    )
    occluded_ids = {word_ids[i] for i in occluded}

    pts = [
        densify(p, step) if closed else densify_open(p, step) for p, closed in shapes
    ]
    background = int(rng.integers(20, 90))  # same "table" for every canvas change
    img, pts = apply_ops(
        img, pts, rng, [n for n in ops if n in GEOMETRIC], cfg, background
    )
    h, w = img.shape[:2]
    final = [finalize_shape(p, w, h, c) for p, (_, c) in zip(pts, shapes, strict=False)]
    ink = dict(zip(word_ids, ink_boxes(img, [final[k] for k in word_k]), strict=False))
    img, _ = apply_ops(
        img, pts, rng, [n for n in ops if n in PHOTOMETRIC], cfg, background
    )

    for el, p in zip(elements, final, strict=False):
        el.set("points", " ".join(f"{x},{y}" for x, y in p))
    page.set("imageWidth", str(w))
    page.set("imageHeight", str(h))

    lines, words, meta = parse_pagexml(io.BytesIO(ET.tostring(tree.getroot())))
    ann = build_annotation(xml_path, lines, words, meta, ink, occluded_ids, level, ops)
    return {
        "image": img,
        "tree": tree,
        "ann": ann,
        "ops": ops,
        "lines": ann_frame(ann, "lines"),
        "words": ann_frame(ann, "words"),
    }


def build_annotation(xml_path, lines, words, meta, ink, occluded_ids, level, ops):
    """Ground truth of one augmented page as a JSON-ready dict.
    Reading order comes from the ORIGINAL page (lines top -> bottom, words left ->
    right), so it does not depend on how the page was warped."""
    o_lines, o_words, _ = parse_pagexml(xml_path)
    o_lines["yc"] = [(b[1] + b[3]) / 2 for b in o_lines["bbox"]]
    line_idx = {lid: i for i, lid in enumerate(o_lines.sort_values("yc")["line_id"])}
    word_idx = {
        wid: i
        for wid, i in zip(
            o_words["word_id"], o_words.groupby("line_id").cumcount(), strict=False
        )
    }

    W = []
    for _, r in words.iterrows():
        W.append(
            {
                "word_id": r["word_id"],
                "line_id": r["line_id"],
                "line_idx": line_idx[r["line_id"]],
                "word_idx": int(word_idx[r["word_id"]]),
                "text": normalize_text(r["text"]),
                "occluded": r["word_id"] in occluded_ids,
                "polygon": np.asarray(r["polygon"]).tolist(),
                "bbox": list(map(int, r["bbox"])),
                "ink_bbox": None
                if ink.get(r["word_id"]) is None
                else list(ink[r["word_id"]]),
            }
        )
    W.sort(key=lambda d: (d["line_idx"], d["word_idx"]))

    L = []
    for _, r in lines.iterrows():
        lw = [d for d in W if d["line_id"] == r["line_id"]]
        vis = [d for d in lw if not d["occluded"]]
        text = normalize_text(r["text"])
        L.append(
            {
                "line_id": r["line_id"],
                "line_idx": line_idx[r["line_id"]],
                "text": text,  # full transcription of the line (TextEquiv)
                # what can actually be read: the line text, or the visible words if
                # some were erased
                "text_visible": text
                if len(vis) == len(lw)
                else " ".join(d["text"] for d in vis),
                "n_words": len(lw),
                "n_visible": len(vis),
                "polygon": np.asarray(r["polygon"]).tolist(),
                "bbox": list(map(int, r["bbox"])),
                "words_bbox": box_union([d["bbox"] for d in vis]),
                "ink_bbox": box_union([d["ink_bbox"] for d in vis]),
            }
        )
    L.sort(key=lambda d: d["line_idx"])
    return {
        "image": meta["image"],
        "width": meta["width"],
        "height": meta["height"],
        "level": level,
        "ops": ops,
        "occluded": sorted(occluded_ids),
        "lines": L,
        "words": W,
    }


def ann_frame(ann, key):
    """Annotation list -> DataFrame (bbox columns as tuples, like parse_pagexml)."""
    df = pd.DataFrame(ann[key])
    for col in ["bbox", "ink_bbox", "words_bbox"]:
        if col in df:
            df[col] = [None if b is None else tuple(b) for b in df[col]]
    return df


def write_pagexml(tree, path):
    ns = tree.getroot().tag.split("}")[0].strip("{")
    ET.register_namespace("", ns)  # keep the PAGE namespace as default (no ns0:)
    tree.write(path, encoding="utf-8", xml_declaration=True)


# --- Plot helpers ---------------------------------------------------------------


def draw_boxes(ax, img, boxes, color, lw, title=None, ls="-"):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    for b in boxes:
        if b is None:
            continue
        x0, y0, x1, y1 = b
        ax.add_patch(
            Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                fill=False,
                edgecolor=color,
                linewidth=lw,
                linestyle=ls,
            )
        )
    if title:
        ax.set_title(title, fontsize=9)
    ax.axis("off")


def demo_pages(xml_path):
    """Every level (original included) for one page, same RNG as the dataset build."""
    return {level: augment_pagexml(xml_path, level) for level in LEVELS}

## Line plots on alterated images
- Red: the box of the warped line outline.
- Blue dashed: the box around the line’s ink.
- Blue numbers: the reading-order index.

In [ ]:
# --- Alteration levels with LINE bounding boxes --------------------------------
DEMO_XML = eparchos_first_xml  # change to look at another page

demo = demo_pages(DEMO_XML)

fig, axes = plt.subplots(1, len(demo), figsize=(6 * len(demo), 7))
for ax, (name, d) in zip(axes, demo.items(), strict=False):
    L = d["lines"]
    ops = ", ".join(d["ops"]) or "none"
    # red: box of the warped line polygon / blue dashed: box around the line's ink
    draw_boxes(
        ax, d["image"], L["bbox"], "red", 1.0, f"{name}  ({len(L)} lines)\n{ops}"
    )
    draw_boxes(ax, d["image"], L["ink_bbox"], "dodgerblue", 0.8, ls="--")
    for i, (x0, y0, _, _) in zip(L["line_idx"], L["bbox"], strict=False):
        ax.text(x0 - 4, y0 + 15, str(i), color="blue", fontsize=7, ha="right")
plt.tight_layout()
plt.show()

## Words plot on altered images
- Green: the ink box of each word, which is the target the benchmark scores word boxes against.
- Red dashed: erased words; there should be no ink inside them.
- Bottom row: a zoom on the lines in ZOOM_LINES (reading-order indices), placed using the warped boxes so it follows the text on tilted pages.

In [ ]:
# --- Alteration levels with WORD bounding boxes --------------------------------
DEMO_XML = eparchos_first_xml
ZOOM_LINES = [0, 1, 2]  # reading-order indices of the lines shown in the zoomed row

demo = demo_pages(DEMO_XML)

fig, axes = plt.subplots(
    2, len(demo), figsize=(6 * len(demo), 10), gridspec_kw={"height_ratios": [3, 1]}
)
for col, (name, d) in enumerate(demo.items()):
    W = d["words"]
    vis, occ = W[~W["occluded"]], W[W["occluded"]]
    title = (
        f"{name}  ({len(W)} words, {len(occ)} erased)\n{', '.join(d['ops']) or 'none'}"
    )
    # green: ink box (evaluation target) / red dashed: erased words (hallucination test)
    for row, sub in [(0, W), (1, W[W["line_idx"].isin(ZOOM_LINES)])]:
        ax = axes[row, col]
        draw_boxes(
            ax,
            d["image"],
            sub.loc[~sub["occluded"], "ink_bbox"],
            "lime",
            0.8,
            title if row == 0 else f"zoom on lines {ZOOM_LINES}",
        )
        draw_boxes(
            ax, d["image"], sub.loc[sub["occluded"], "bbox"], "red", 1.2, ls="--"
        )
    z = W[W["line_idx"].isin(ZOOM_LINES)]
    x0, y0 = np.min([b[:2] for b in z["bbox"]], axis=0) - 10
    x1, y1 = np.max([b[2:] for b in z["bbox"]], axis=0) + 10
    axes[1, col].set_xlim(max(x0, 0), x1)
    axes[1, col].set_ylim(y1, max(y0, 0))
plt.tight_layout()
plt.show()

In [ ]:
# --- Build eparchos_aug/{original, light, moderate, heavy} (skips what exists) --
FORCE_REBUILD = False  # True = wipe and regenerate the levels

src_xml_dir = EPARCHOS_ROOT / "eparchos" / "xml"
src_img_dir = EPARCHOS_ROOT / "eparchos" / "image"
src_xmls = sorted(src_xml_dir.glob("*.xml"))
assert src_xmls, f"no XML found in {src_xml_dir}: run the download cell first"
print(f"{len(src_xmls)} source pages in {src_xml_dir}")


def level_is_complete(level_dir, level):
    manifest = level_dir / "manifest.json"
    if not manifest.exists():
        return False
    m = json.loads(manifest.read_text())
    if m.get("fingerprint") != level_fingerprint(level):
        raise RuntimeError(
            f"{level_dir} was built with another config/seed/version. "
            "Set FORCE_REBUILD = True to regenerate it."
        )
    n_ann = len(list((level_dir / "ann").glob("*.json")))
    return m.get("n_pages") == len(src_xmls) == n_ann


def flat(records, box_cols):
    """Annotation records -> flat table: one x0..y1 column group per box."""
    df = pd.DataFrame(records).drop(columns=["polygon"])
    for col, prefix in box_cols.items():
        boxes = [b if b is not None else [None] * 4 for b in df.pop(col)]
        df[[f"{prefix}{c}" for c in ("x0", "y0", "x1", "y1")]] = pd.DataFrame(
            boxes, index=df.index
        )
    return df


for level in LEVELS:
    level_dir = AUG_DIR / level
    if FORCE_REBUILD and level_dir.exists():
        import shutil

        shutil.rmtree(level_dir)

    if level_is_complete(level_dir, level):
        print(f"[{level}] already built ({len(src_xmls)} pages) -> skip")
        continue

    for sub in ("image", "xml", "ann"):
        (level_dir / sub).mkdir(parents=True, exist_ok=True)

    anns, errors = [], []
    for xml_path in tqdm(src_xmls, desc=level):
        out_ann = level_dir / "ann" / (xml_path.stem + ".json")
        try:
            if out_ann.exists():  # resume an interrupted build (ann is written last)
                anns.append(json.loads(out_ann.read_text()))
                continue
            image_name = (
                ET.parse(xml_path).getroot().find(".//{*}Page").get("imageFilename")
            )
            out_img = level_dir / "image" / (Path(image_name).stem + ".png")
            r = augment_pagexml(xml_path, level, image_dir=src_img_dir)
            r["tree"].getroot().find(".//{*}Page").set("imageFilename", out_img.name)
            r["ann"]["image"] = out_img.name
            r["ann"]["xml"] = xml_path.name
            Image.fromarray(r["image"]).save(out_img)
            write_pagexml(r["tree"], level_dir / "xml" / xml_path.name)
            out_ann.write_text(json.dumps(r["ann"], ensure_ascii=False))
            anns.append(r["ann"])
        except Exception as e:  # one bad page must not kill the whole build
            errors.append((xml_path.name, repr(e)))

    boxes = {"bbox": "", "ink_bbox": "ink_"}
    lines = pd.concat(
        [
            flat(a["lines"], {**boxes, "words_bbox": "words_"}).assign(image=a["image"])
            for a in anns
        ]
    )
    words = pd.concat([flat(a["words"], boxes).assign(image=a["image"]) for a in anns])
    lines.to_csv(level_dir / "lines.csv", index=False)
    words.to_csv(level_dir / "words.csv", index=False)
    pd.DataFrame(
        [
            {
                "image": a["image"],
                "xml": a.get("xml"),
                "width": a["width"],
                "height": a["height"],
                "ops": "|".join(a["ops"]),
                "occluded": "|".join(a["occluded"]),
            }
            for a in anns
        ]
    ).to_csv(level_dir / "pages.csv", index=False)

    if errors:  # no manifest -> the next run retries the failed pages
        print(f"[{level}] {len(errors)} page(s) failed, e.g. {errors[:3]}")
        continue
    (level_dir / "manifest.json").write_text(
        json.dumps(
            {
                "level": level,
                "seed": AUG_SEED,
                "version": PIPELINE_VERSION,
                "fingerprint": level_fingerprint(level),
                "n_pages": len(anns),
                "config": LEVELS[level],
            },
            indent=2,
            default=str,
        )
    )
    print(
        f"[{level}] done: {len(anns)} pages, {len(lines)} lines, {len(words)} words, "
        f"{int(words['occluded'].sum())} erased"
    )

print("\nDataset at", AUG_DIR.resolve())
for level in LEVELS:
    d = AUG_DIR / level
    print(
        f"  {level:9s} {len(list((d / 'image').glob('*.png'))):5d} images  "
        f"{len(list((d / 'ann').glob('*.json'))):5d} annotations"
    )

# Evaluation

In [ ]:
from collections import Counter

from scipy.optimize import linear_sum_assignment

try:
    from rapidfuzz.distance import Levenshtein as _Lev

    def edit_distance(a, b):
        return _Lev.distance(a, b)
except ImportError:

    def edit_distance(a, b):  # works on strings and on lists of words
        prev = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            cur = [i]
            for j, cb in enumerate(b, 1):
                cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
            prev = cur
        return prev[-1]


IOU_THR = 0.5  # a line / word counts as found above this overlap
TEXT_PAIR_THR = 0.25  # looser pairing used only to compare line TEXTS end-to-end, so a
# slightly misplaced line is not also counted as a missing text
LINE_GT_SHAPE = (
    "words"  # vs predicted polygons. "words": union of the line's word polygons
)
# (tight, follows tilt) / "line": annotated line polygon (warped)
LINE_GT_BOX = (
    "ink_bbox"  # vs predicted boxes: "ink_bbox" (box around the ink) or "words_bbox"
)
WORD_GT_BOX = (
    "ink_bbox"  # "ink_bbox" (tight on ink) or "bbox" (annotated rectangle, warped)
)
PRED_DIR = AUG_DIR / "predictions"
MODES = ("lines", "text_oracle", "read_oracle", "words_oracle", "e2e")


# --- Ground truth ----------------------------------------------------------------


def load_page(level, stem):
    ann = json.loads((AUG_DIR / level / "ann" / f"{stem}.json").read_text())
    return load_page_image(AUG_DIR / level / "image" / ann["image"]), ann


def gt_lines(ann):
    """Lines that still have readable words, in reading order."""
    return [line for line in ann["lines"] if line["n_visible"] > 0]


def gt_line_input(ann):
    """The true lines in the format a model receives (for the oracle modes)."""
    return [
        {"bbox": line["bbox"], "polygon": line["polygon"]} for line in gt_lines(ann)
    ]  # noqa: E501


def gt_words(ann):
    return [w for w in ann["words"] if not w["occluded"]]


def word_gt_box(w):
    return w[WORD_GT_BOX] if w[WORD_GT_BOX] is not None else w["bbox"]


# --- Overlaps ----------------------------------------------------------------------


def rasterize(polys):
    """Polygons -> (x0, y0, boolean mask cropped to their box)."""
    polys = [np.asarray(p, np.float32).reshape(-1, 2) for p in polys]
    allp = np.concatenate(polys)
    x0, y0 = np.floor(allp.min(axis=0)).astype(int)
    x1, y1 = np.ceil(allp.max(axis=0)).astype(int) + 1
    m = np.zeros((y1 - y0, x1 - x0), np.uint8)
    cv2.fillPoly(m, [np.round(p - [x0, y0]).astype(np.int32) for p in polys], 1)
    return int(x0), int(y0), m.astype(bool)


def rect(b):
    x0, y0, x1, y1 = b
    return [[x0, y0], [x1, y0], [x1, y1], [x0, y1]]


def gt_line_mask(ann, line):
    if LINE_GT_SHAPE == "line":
        return rasterize([line["polygon"]])
    return rasterize(
        [w["polygon"] for w in gt_words(ann) if w["line_id"] == line["line_id"]]
    )


def pred_line_mask(line):
    if line.get("polygons"):
        return rasterize(line["polygons"])
    return rasterize(
        [line["polygon"] if line.get("polygon") is not None else rect(line["bbox"])]
    )


def mask_overlap(a, b):
    (ax, ay, am), (bx, by, bm) = a, b
    x0, y0 = max(ax, bx), max(ay, by)
    x1, y1 = (
        min(ax + am.shape[1], bx + bm.shape[1]),
        min(ay + am.shape[0], by + bm.shape[0]),
    )
    if x1 <= x0 or y1 <= y0:
        return 0
    return int(
        np.logical_and(
            am[y0 - ay : y1 - ay, x0 - ax : x1 - ax],
            bm[y0 - by : y1 - by, x0 - bx : x1 - bx],
        ).sum()
    )


def box_iou_matrix(A, B):
    A, B = np.asarray(A, float).reshape(-1, 4), np.asarray(B, float).reshape(-1, 4)
    if not len(A) or not len(B):
        return np.zeros((len(A), len(B)))
    ix = np.clip(
        np.minimum(A[:, None, 2], B[None, :, 2])
        - np.maximum(A[:, None, 0], B[None, :, 0])
        + 1,
        0,
        None,
    )
    iy = np.clip(
        np.minimum(A[:, None, 3], B[None, :, 3])
        - np.maximum(A[:, None, 1], B[None, :, 1])
        + 1,
        0,
        None,
    )
    inter = ix * iy

    def area(X):
        return (X[:, 2] - X[:, 0] + 1) * (X[:, 3] - X[:, 1] + 1)

    return inter / (area(A)[:, None] + area(B)[None, :] - inter)


def match(iou, thr=IOU_THR):
    """One-to-one matching maximising overlap (Hungarian), pairs with iou >= thr."""
    if iou.size == 0:
        return []
    rows, cols = linear_sum_assignment(-iou)
    return [
        (int(r), int(c), float(iou[r, c]))
        for r, c in zip(rows, cols, strict=False)
        if iou[r, c] >= thr
    ]


# --- Stage scores (raw counts, summed over pages afterwards) ------------------------


def box_area(b):
    return (b[2] - b[0] + 1) * (b[3] - b[1] + 1)


def box_inter(a, b):
    return max(0, min(a[2], b[2]) - max(a[0], b[0]) + 1) * max(
        0, min(a[3], b[3]) - max(a[1], b[1]) + 1
    )


def line_gt_box(line):
    return line[LINE_GT_BOX] if line[LINE_GT_BOX] is not None else line["words_bbox"]


def line_overlaps(ann, pred_lines):
    """Intersection / areas between true and predicted lines. A prediction with a
    polygon is compared with the true line SHAPE (masks); a box-only prediction
    with the true line BOX - a box cannot follow a tilted line, so comparing it
    with a thin tilted shape would punish the format, not the model."""
    G = gt_lines(ann)
    inter = np.zeros((len(G), len(pred_lines)))
    ga = np.zeros_like(inter)
    pa = np.zeros(len(pred_lines))
    gmasks = None
    for j, pl in enumerate(pred_lines):
        if pl.get("polygons") or pl.get("polygon") is not None:
            gmasks = gmasks or [gt_line_mask(ann, line) for line in G]
            pm = pred_line_mask(pl)
            pa[j] = pm[2].sum()
            for i, g in enumerate(gmasks):
                inter[i, j], ga[i, j] = mask_overlap(g, pm), g[2].sum()
        else:
            pa[j] = box_area(pl["bbox"])
            for i, line in enumerate(G):
                inter[i, j], ga[i, j] = (
                    box_inter(line_gt_box(line), pl["bbox"]),
                    box_area(line_gt_box(line)),
                )
    return inter, ga, pa


def score_lines(ann, pred_lines):
    """Returns detection counts and the line pairs used to score the text."""
    inter, ga, pa = line_overlaps(ann, pred_lines)
    iou = inter / (ga + pa[None, :] - inter + 1e-9)
    pairs = match(iou)
    g_of = {p: g for g, p, _ in pairs}
    p_of = {g: p for g, p, _ in pairs}
    # merge: one prediction covers most of 2+ true lines that have no match of their own
    # split: one true line holds 2+ predictions that have no match of their own
    # (overlap alone is not enough: tilted boxes always overlap their neighbours)
    cover_g = inter / (ga + 1e-9) > 0.5
    cover_p = inter / (pa[None, :] + 1e-9) > 0.5
    merges = sum(
        sum(p_of.get(g, p) == p for g in np.nonzero(cover_g[:, p])[0]) >= 2
        for p in range(inter.shape[1])
    )
    splits = sum(
        sum(g_of.get(p, g) == g for p in np.nonzero(cover_p[g])[0]) >= 2
        for g in range(inter.shape[0])
    )
    counts = {
        "n_gt": inter.shape[0],
        "n_pred": inter.shape[1],
        "matched": len(pairs),
        "iou_sum": sum(p[2] for p in pairs),
        "merges": merges,
        "splits": splits,
    }
    return counts, match(iou, TEXT_PAIR_THR)


def score_text(gt_texts, pred_texts, pairs, occluded_by_line):
    """pairs: (gt index, pred index). Missed true lines cost all their characters,
    extra predicted lines cost all theirs."""
    s = Counter()
    got = {g: p for g, p, *_ in pairs}
    for g, gt in enumerate(gt_texts):
        gt = normalize_text(gt)
        pr = normalize_text(pred_texts[got[g]]) if g in got else ""
        s["char_edits"] += edit_distance(gt, pr)
        s["chars"] += len(gt)
        s["word_edits"] += edit_distance(gt.split(), pr.split())
        s["words"] += len(gt.split())
        # hallucination: an erased word appears more often than the visible text allows
        seen, allowed = (
            Counter(map(strip_punct, pr.split())),
            Counter(map(strip_punct, gt.split())),
        )
        for w in occluded_by_line.get(g, []):
            s["occluded"] += 1
            if seen[strip_punct(w)] > allowed[strip_punct(w)]:
                s["hallucinated"] += 1
                seen[strip_punct(w)] -= 1
    for p in set(range(len(pred_texts))) - set(got.values()):
        extra = normalize_text(pred_texts[p])
        s["char_edits"] += len(extra)
        s["word_edits"] += len(extra.split())
    return dict(s)


def score_words(ann, pred_words):
    """All predicted words of the page against all visible true words."""
    G = gt_words(ann)
    P = [w for line in pred_words for w in line]
    iou = box_iou_matrix([word_gt_box(w) for w in G], [w["bbox"] for w in P])
    pairs = match(iou)
    correct = sum(
        normalize_text(G[g]["text"]) == normalize_text(P[p]["text"])
        for g, p, _ in pairs
    )
    return {
        "n_gt": len(G),
        "n_pred": len(P),
        "matched": len(pairs),
        "iou_sum": sum(p[2] for p in pairs),
        "correct": int(correct),
    }


def occluded_words_per_line(ann):
    """Erased words used for the hallucination test, per line. Only words that
    appear nowhere else on the page: if the same word is visible on another line
    (καὶ, τὸν...), a reader that grabs a bit of the neighbouring line would be
    counted as inventing it."""
    idx = {line["line_id"]: i for i, line in enumerate(gt_lines(ann))}
    visible = Counter(strip_punct(w["text"]) for w in gt_words(ann))
    out = {}
    for w in ann["words"]:
        if (
            w["occluded"]
            and w["line_id"] in idx
            and visible[strip_punct(w["text"])] == 0
        ):
            out.setdefault(idx[w["line_id"]], []).append(w["text"])
    return out


# --- Running a model -----------------------------------------------------------------


def _jsonable(o):
    return (
        o.tolist()
        if hasattr(o, "tolist")
        else int(o)
        if isinstance(o, np.integer)
        else float(o)
    )


def cached(model, level, stem, mode, fn):
    if not getattr(model, "cache", False):
        return fn()
    path = PRED_DIR / model.name / level / f"{stem}.{mode}.json"
    if path.exists():
        return json.loads(path.read_text())
    out = fn()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(out, ensure_ascii=False, default=_jsonable))
    return out


def available_modes(model):
    def has(m):
        return callable(getattr(model, m, None))

    return [
        m
        for m, ok in {
            "lines": has("find_lines"),
            "text_oracle": has("transcribe_lines"),
            "read_oracle": has("read_lines"),
            "words_oracle": has("find_words"),
            "e2e": has("predict")
            or all(map(has, ("find_lines", "transcribe_lines", "find_words"))),
        }.items()
        if ok
    ]


def run_e2e(model, image):
    if callable(getattr(model, "predict", None)):
        return list(model.predict(image))
    lines = model.find_lines(image)
    texts = model.transcribe_lines(image, lines)
    return [lines, texts, model.find_words(image, lines, texts)]


def evaluate_page(model, level, stem, modes=MODES):
    image, ann = load_page(level, stem)
    if hasattr(model, "peek_ground_truth"):  # only the oracle sanity-check model
        model.peek_ground_truth(ann)
    G = gt_lines(ann)
    gt_texts = [line["text_visible"] for line in G]
    occl = occluded_words_per_line(ann)
    row = {"model": model.name, "level": level, "page": stem}
    todo = [m for m in modes if m in available_modes(model)]

    if "lines" in todo:
        lines = cached(model, level, stem, "lines", lambda: model.find_lines(image))
        s, _ = score_lines(ann, lines)
        row.update({f"lines/{k}": v for k, v in s.items()})
    if "text_oracle" in todo:
        texts = cached(
            model,
            level,
            stem,
            "text_oracle",
            lambda: model.transcribe_lines(image, gt_line_input(ann)),
        )
        s = score_text(
            gt_texts, texts, [(i, i) for i in range(min(len(G), len(texts)))], occl
        )
        row.update({f"text_oracle/{k}": v for k, v in s.items()})
    if "read_oracle" in todo:
        texts, words = cached(
            model,
            level,
            stem,
            "read_oracle",
            lambda: model.read_lines(image, gt_line_input(ann)),
        )
        s = score_text(
            gt_texts, texts, [(i, i) for i in range(min(len(G), len(texts)))], occl
        )
        row.update({f"read_oracle/{k}": v for k, v in s.items()})
        row.update(
            {f"read_oracle/words_{k}": v for k, v in score_words(ann, words).items()}
        )
    if "words_oracle" in todo:
        words = cached(
            model,
            level,
            stem,
            "words_oracle",
            lambda: model.find_words(image, gt_line_input(ann), gt_texts),
        )
        row.update({f"words_oracle/{k}": v for k, v in score_words(ann, words).items()})
    if "e2e" in todo:
        lines, texts, words = cached(
            model, level, stem, "e2e", lambda: run_e2e(model, image)
        )
        texts = list(texts)[: len(lines)] + [""] * (
            len(lines) - len(texts)
        )  # one text per line
        s, pairs = score_lines(ann, lines)
        row.update({f"e2e/lines_{k}": v for k, v in s.items()})
        row.update(
            {f"e2e/{k}": v for k, v in score_text(gt_texts, texts, pairs, occl).items()}
        )
        row.update({f"e2e/words_{k}": v for k, v in score_words(ann, words).items()})
    return row


def benchmark_pages(n=None):
    """All pages, or a fixed subset of n pages saved next to the dataset so every
    model is scored on the same pages."""
    stems = sorted(p.stem for p in (AUG_DIR / "original" / "ann").glob("*.json"))
    if n is None:
        return stems
    path = AUG_DIR / f"benchmark_subset_{n}.txt"
    if not path.exists():
        path.write_text(
            "\n".join(sorted(stems, key=lambda s: zlib.crc32(s.encode()))[:n])
        )
    return path.read_text().split("\n")


def run_benchmark(models, levels, pages, modes=MODES):
    rows = []
    for model in models:
        for level in levels:
            for stem in tqdm(pages, desc=f"{model.name} / {level}"):
                try:
                    rows.append(evaluate_page(model, level, stem, modes))
                except Exception as e:
                    rows.append(
                        {
                            "model": model.name,
                            "level": level,
                            "page": stem,
                            "error": repr(e),
                        }
                    )
    return pd.DataFrame(rows)


def summarize(results):
    """Pages summed per model x level (micro average), then turned into rates."""

    def ratio(a, b):
        return a / b if b else np.nan

    def f1(p, r):
        return 2 * p * r / (p + r) if p + r else 0.0

    out = []
    num = results.select_dtypes("number").columns
    for (model, level), g in results.groupby(["model", "level"], sort=False):
        s = g[num].sum(min_count=1)

        def get(k):
            return s.get(k, np.nan)  # noqa: B023

        row = {
            "model": model,
            "level": level,
            "pages": len(g),
            "errors": int(g["error"].notna().sum()) if "error" in g else 0,
        }
        for pre, name in [("lines/", "1 lines"), ("e2e/lines_", "e2e lines")]:
            p, r = (
                ratio(get(pre + "matched"), get(pre + "n_pred")),
                ratio(get(pre + "matched"), get(pre + "n_gt")),
            )
            row |= {
                f"{name} F1": f1(p, r),
                f"{name} mIoU": ratio(get(pre + "iou_sum"), get(pre + "matched")),
                f"{name} merges": get(pre + "merges"),
                f"{name} splits": get(pre + "splits"),
            }
        for pre, name in [
            ("text_oracle/", "2 text|true lines"),
            ("read_oracle/", "2+3 read|true lines"),
            ("e2e/", "e2e text"),
        ]:
            row |= {
                f"{name} CER": ratio(get(pre + "char_edits"), get(pre + "chars")),
                f"{name} WER": ratio(get(pre + "word_edits"), get(pre + "words")),
                f"{name} hallucinated": ratio(
                    get(pre + "hallucinated"), get(pre + "occluded")
                ),
            }
        for pre, name in [
            ("words_oracle/", "3 words|true lines+text"),
            ("read_oracle/words_", "2+3 words|true lines"),
            ("e2e/words_", "e2e words"),
        ]:
            p, r = (
                ratio(get(pre + "matched"), get(pre + "n_pred")),
                ratio(get(pre + "matched"), get(pre + "n_gt")),
            )
            pc, rc = (
                ratio(get(pre + "correct"), get(pre + "n_pred")),
                ratio(get(pre + "correct"), get(pre + "n_gt")),
            )
            row |= {
                f"{name} F1": f1(p, r),
                f"{name} mIoU": ratio(get(pre + "iou_sum"), get(pre + "matched")),
                f"{name} F1 (box + text right)": f1(pc, rc),
            }
        out.append(row)
    return pd.DataFrame(out).dropna(axis=1, how="all").set_index(["model", "level"])

In [ ]:
# --- Benchmark only on pages no model has seen: the TEST split --------------------
# train: used to train kraken_eparchos / val: used to tune it (e.g. words_in_line)
# -> only `test` is fair. Save the split JSON at SPLIT_FILE (or change the path).
SPLIT_FILE = "../data/eparchos_split.json"
BENCH_SPLIT = "test"

SPLIT = json.loads(Path(SPLIT_FILE).read_text())
all_split = [p for pages_ in SPLIT.values() for p in pages_]
assert len(all_split) == len(set(all_split)), "a page is in two splits"
available = {p.stem for p in (AUG_DIR / "original" / "ann").glob("*.json")}
missing = set(SPLIT[BENCH_SPLIT]) - available
assert not missing, (
    f"{len(missing)} {BENCH_SPLIT} pages are not in the dataset, e.g. {sorted(missing)[:3]}"  # noqa: E501
)
print(
    {k: len(v) for k, v in SPLIT.items()},
    f"-> benchmark on the {len(SPLIT[BENCH_SPLIT])} {BENCH_SPLIT} pages",
)  # noqa: E501
if available - set(all_split):
    print(f"note: {len(available - set(all_split))} dataset pages are in no split")


def benchmark_pages(n=None, split=None):  # noqa: F811
    """Pages of the benchmark split (test by default), or a fixed subset of n of
    them (same pages for every model). Replaces the version of the evaluation cell."""
    stems = sorted(SPLIT[split or BENCH_SPLIT])
    if n is None:
        return stems
    return sorted(sorted(stems, key=lambda s: zlib.crc32(s.encode()))[:n])

In [ ]:
# --- Models to benchmark -------------------------------------------------------


class OracleModel:
    """Cheats by reading the ground truth. Must score perfectly: if it does not,
    the evaluation itself is wrong. Run it once after any change to the pipeline."""

    name = "oracle (sanity check)"

    def peek_ground_truth(self, ann):
        self.ann = ann

    def find_lines(self, image):
        return [
            {
                "bbox": line["bbox"],
                "polygons": [
                    w["polygon"]
                    for w in gt_words(self.ann)
                    if w["line_id"] == line["line_id"]
                ],
            }
            for line in gt_lines(self.ann)
        ]

    def transcribe_lines(self, image, lines):
        return [line["text_visible"] for line in gt_lines(self.ann)]

    def find_words(self, image, lines, texts):
        return [
            [
                {"text": w["text"], "bbox": word_gt_box(w)}
                for w in gt_words(self.ann)
                if w["line_id"] == line["line_id"]
            ]
            for line in gt_lines(self.ann)
        ]


class ProportionalWords:
    """Stage 3 baseline: cut each line box into slices proportional to the number
    of letters of each word, then shrink each slice to its ink (what the
    kraken + Claude predictor does for words it cannot match). Assumes straight
    horizontal lines, so tilt and curl should hurt it."""

    name = "proportional split (stage 3)"

    def find_words(self, image, lines, texts):
        gray = to_gray(image)
        out = []
        for line, text in zip(lines, texts, strict=False):
            x0, y0, x1, y1 = line["bbox"]
            crop = gray[y0 : y1 + 1, x0 : x1 + 1]
            thr = (
                cv2.threshold(
                    crop.reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
                )[0]
                if crop.size
                else 0
            )
            tokens = normalize_text(text).split()
            weights = np.array([len(t) + 1 for t in tokens], float)
            edges = x0 + np.r_[0, np.cumsum(weights)] / max(weights.sum(), 1) * (
                x1 - x0
            )
            words = []
            for t, a, b in zip(tokens, edges[:-1], edges[1:], strict=False):
                a, b = int(a), max(int(b), int(a) + 1)
                ys, xs = np.nonzero(gray[y0 : y1 + 1, a : b + 1] <= thr)
                box = (
                    [a + xs.min(), y0 + ys.min(), a + xs.max(), y0 + ys.max()]
                    if len(xs)
                    else [a, y0, b, y1]
                )
                words.append({"text": t, "bbox": [int(v) for v in box]})
            out.append(words)
        return out


class WordPredictorAdapter:
    """Wraps one of your existing predictors (KrakenPredictor, ClaudeCodePredictor:
    predict(image bytes) -> list[Prediction] with box = (x, y, w, h) and a line
    number). They are end-to-end only, so only the e2e mode is scored."""

    def __init__(self, predictor, name=None, cache=True):
        self.predictor = predictor
        self.name = name or type(predictor).__name__
        self.cache = cache

    def predict(self, image):
        buf = io.BytesIO()
        Image.fromarray(image).save(buf, format="PNG")
        found = self.predictor.predict(buf.getvalue())
        by_line = {}
        for p in found:
            x, y, w, h = p.box
            by_line.setdefault(p.line, []).append(
                {"text": p.text, "bbox": [x, y, x + w - 1, y + h - 1]}
            )
        words = [by_line[k] for k in sorted(by_line)]
        lines = [{"bbox": list(box_union([w["bbox"] for w in ws]))} for ws in words]
        texts = [" ".join(w["text"] for w in ws) for ws in words]
        return lines, texts, words

In [ ]:
# --- Sanity check: the oracle must score perfectly --------------------------------
SANITY_PAGES = benchmark_pages()[:5]

sanity = run_benchmark([OracleModel(), ProportionalWords()], list(LEVELS), SANITY_PAGES)
sanity_summary = summarize(sanity)

oracle = sanity_summary.loc["oracle (sanity check)"]
perfect = {c: 1.0 for c in oracle.columns if c.endswith(("F1", "mIoU", "right)"))}
perfect |= {
    c: 0.0
    for c in oracle.columns
    if c.endswith(("CER", "WER", "merges", "splits", "hallucinated"))
}
wrong = {
    c: oracle[c].tolist()
    for c, v in perfect.items()
    if not np.allclose(oracle[c].fillna(v), v)
}
assert not wrong, f"the evaluation is broken, the oracle is not perfect on: {wrong}"
print(f"oracle perfect on {len(SANITY_PAGES)} pages x {len(LEVELS)} levels")

with pd.option_context(
    "display.max_columns", None, "display.width", 250, "display.precision", 3
):
    display(sanity_summary.T)

## Kraken evaluation

In [ ]:
# --- Kraken, step by step ---------------------------------------------------------
# import sys; sys.path.insert(0, "path/to/backend")  # if `ml` is not importable here
from ml.kraken_predictor import (
    KrakenBasePredictor,
    KrakenPredictor,
    bounds,
    rectangle,
    straight_lines,
    words_in_line,
)


def xywh_to_xyxy(box):
    """Prediction.box (x, y, w, h) -> benchmark box [x0, y0, x1, y1] (inclusive)."""
    x, y, w, h = box
    return [int(x), int(y), int(x + w - 1), int(y + h - 1)]


def file_hash(path, n=8):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:n]


class KrakenStages:
    """Runs a KrakenPredictor step by step, with the same calls as its predict():
      find_lines  -> the segmenter's line polygons (what step 1 really finds)
      read_lines  -> each line turned into its bounding rectangle with a straight
                     baseline (as predict() does), recognised, words cut from the
                     reading with words_in_line()
      predict     -> find_lines then read_lines, lines without words dropped:
                     the same output as KrakenPredictor.predict (checked below)
    The name includes a hash of the weights, so cached predictions are not reused
    after the model is retrained."""

    def __init__(self, predictor_class=KrakenPredictor, name=None, cache=True):
        self.kraken = predictor_class()
        weights = predictor_class.weights_path()
        self.name = f"{name or self.kraken.model_version}-{file_hash(weights)}"
        self.cache = cache
        self._last = (
            None,
            None,
        )  # segmentation of the last image (lines + e2e share it)

    def _segment(self, image):
        from kraken.configs import SegmentationInferenceConfig

        key = hashlib.md5(image.tobytes()).hexdigest()
        if self._last[0] != key:
            with self.kraken.lock:
                found = self.kraken.segmenter.predict(
                    Image.fromarray(image),
                    SegmentationInferenceConfig(accelerator="cpu"),
                )
            self._last = (key, [line.boundary for line in found.lines])
        return self._last[1]

    def find_lines(self, image):
        h, w = image.shape[:2]
        return [
            {"bbox": list(bounds(b, w, h)), "polygon": [[int(x), int(y)] for x, y in b]}
            for b in self._segment(image)
        ]

    def read_lines(self, image, lines):
        """Text and words of each given line, in the order of `lines` ("" / [] if
        nothing is read on a line)."""
        from kraken.configs import RecognitionInferenceConfig

        h, w = image.shape[:2]
        page = Image.fromarray(image)
        gray = np.asarray(page.convert("L"), dtype=np.uint8)
        rects = [
            rectangle(
                line["polygon"]
                if line.get("polygon") is not None
                else rect(line["bbox"]),
                w,
                h,
            )
            for line in lines
        ]
        texts, words = [""] * len(lines), [[] for _ in lines]
        if not rects:
            return texts, words
        with self.kraken.lock:
            records = list(
                self.kraken.recognizer.predict(
                    page,
                    straight_lines(rects),
                    RecognitionInferenceConfig(accelerator="cpu", num_line_workers=0),
                )
            )
        for k, record in enumerate(records):
            rid = str(getattr(record, "id", ""))
            i = (
                int(rid) if rid.isdigit() else k
            )  # straight_lines numbers the lines 0..n-1
            found = words_in_line(
                record.prediction,
                record.cuts,
                record.confidences,
                bounds(record.boundary, w, h),
                gray,
                i,
            )
            words[i] = [
                {
                    "text": p.text,
                    "bbox": xywh_to_xyxy(p.box),
                    "confidence": float(p.probabilities.get(p.text, np.nan)),
                }
                for p in found
            ]
            texts[i] = " ".join(p.text for p in found)
        return texts, words

    def predict(self, image):
        lines = self.find_lines(image)
        texts, words = self.read_lines(image, lines)
        keep = [
            i for i, ws in enumerate(words) if ws
        ]  # predict() drops lines without words
        return (
            [lines[i] for i in keep],
            [texts[i] for i in keep],
            [words[i] for i in keep],
        )

In [ ]:
# --- Load the models and check the wrapper against KrakenPredictor.predict -------
kraken = KrakenStages(KrakenPredictor, name="kraken")
kraken_base = KrakenStages(KrakenBasePredictor, name="kraken-base")

CHECK_PAGES = benchmark_pages()[:2]  # a few pages, on the clean and the heaviest level
for level in ["original", "heavy"]:
    for stem in CHECK_PAGES:
        image, _ = load_page(level, stem)
        buf = io.BytesIO()
        Image.fromarray(image).save(buf, format="PNG")
        real = [
            (p.text, xywh_to_xyxy(p.box), p.line)
            for p in kraken.kraken.predict(buf.getvalue())
        ]
        _, _, words = kraken.predict(image)
        ours = [(w["text"], w["bbox"], i) for i, ws in enumerate(words) for w in ws]
        status = "identical" if ours == real else "DIFFERENT"
        print(
            f"{level:8s} {stem}: {len(real)} words from predict(),\
                 {len(ours)} from the wrapper -> {status}"
        )
        assert ours == real, (
            "the step-by-step wrapper no longer matches KrakenPredictor.predict"
        )

In [ ]:
# --- Benchmark kraken and kraken-base ----------------------------------------------
BENCH_SUBSET = 20  # start small (about 15 s per page and level); None = every page
BENCH_LEVELS = list(LEVELS)  # original, light, moderate, heavy

pages = benchmark_pages(BENCH_SUBSET)
kraken_results = run_benchmark([kraken, kraken_base], BENCH_LEVELS, pages)

out_dir = AUG_DIR / "results"
out_dir.mkdir(exist_ok=True)
kraken_results.to_csv(out_dir / "kraken_per_page.csv", index=False)
kraken_summary = summarize(kraken_results)
kraken_summary.to_csv(out_dir / "kraken_summary.csv")

tested = (
    kraken_results.groupby(["model", "level"], sort=False)["e2e/occluded"].sum(
        min_count=1
    )
    if "e2e/occluded" in kraken_results
    else None
)
print(
    "erased words used for the hallucination test (e2e):",
    "none" if tested is None else tested.dropna().astype(int).to_dict(),
)
if "error" in kraken_results and kraken_results["error"].notna().any():
    print(
        "errors:",
        kraken_results.loc[
            kraken_results["error"].notna(), ["model", "level", "page", "error"]
        ].head(),
    )
with pd.option_context(
    "display.max_columns", None, "display.width", 250, "display.precision", 3
):
    display(kraken_summary.T)

In [ ]:
# --- Look at one page: kraken's lines and words against the ground truth ---------
VIEW_MODEL = kraken
VIEW_PAGE = pages[0]
SHOW_LINES = 4  # first lines whose text is printed under the figure

fig, axes = plt.subplots(1, len(BENCH_LEVELS), figsize=(6 * len(BENCH_LEVELS), 7))
texts_by_level = {}
for ax, level in zip(axes, BENCH_LEVELS, strict=False):
    image, ann = load_page(level, VIEW_PAGE)
    lines, texts, words = cached(
        VIEW_MODEL,
        level,
        VIEW_PAGE,
        "e2e",
        lambda: run_e2e(VIEW_MODEL, image),  # noqa: B023
    )
    r = kraken_results[
        (kraken_results["model"] == VIEW_MODEL.name)
        & (kraken_results["level"] == level)
        & (kraken_results["page"] == VIEW_PAGE)
    ].iloc[0]
    cer = r["e2e/char_edits"] / r["e2e/chars"]
    # green: true word ink boxes / orange: kraken's words / blue: kraken's line polygons
    draw_boxes(
        ax,
        image,
        [word_gt_box(w) for w in gt_words(ann)],
        "lime",
        0.6,
        f"{level}: {len(lines)} lines found / \
            {len(gt_lines(ann))} true, page CER {cer:.1%}",
    )
    draw_boxes(ax, image, [w["bbox"] for ws in words for w in ws], "orangered", 0.6)
    for line in lines:
        p = np.vstack([line["polygon"], line["polygon"][:1]])
        ax.plot(p[:, 0], p[:, 1], color="dodgerblue", linewidth=0.7)
    texts_by_level[level] = (gt_lines(ann), lines, texts)
plt.tight_layout()
plt.show()

# text of the first lines, paired the same way as in the evaluation
for level, (G, lines, texts) in texts_by_level.items():
    image, ann = load_page(level, VIEW_PAGE)
    _, pairs = score_lines(ann, lines)
    got = {g: p for g, p, _ in pairs}
    print(f"--- {level}")
    for g in range(min(SHOW_LINES, len(G))):
        print(f"  true : {G[g]['text_visible']}")
        print(f"  read : {texts[got[g]] if g in got else '(line not found)'}")

## What augmentation hurt the model the most?

In [ ]:
# --- Which augmentation hurts the reading most? ------------------------------------
# Per page: reading error on the TRUE lines (read_oracle), so line finding is out of
# the picture. For each augmentation: error on pages WITH it vs pages WITHOUT it,
# inside the same level (pages of one level differ only by which augmentations fired).
ANALYSIS_MODEL = kraken
ANALYSIS_LEVELS = ["light", "moderate"]  # heavy is ~100% everywhere, nothing to compare

df = kraken_results[kraken_results["model"] == ANALYSIS_MODEL.name].copy()
df["ops"] = [
    load_page(line, p)[1]["ops"]
    for line, p in zip(df["level"], df["page"], strict=False)
]

rows = []
for level in ANALYSIS_LEVELS:
    d = df[df["level"] == level]
    for op in sorted({o for ops in d["ops"] for o in ops}):
        has = d["ops"].apply(lambda ops: op in ops)  # noqa: B023
        if has.all() or not has.any():
            continue  # always or never applied at this level: no comparison possible

        def cer(x):
            return x["read_oracle/char_edits"].sum() / x["read_oracle/chars"].sum()

        rows.append(
            {
                "level": level,
                "augmentation": op,
                "pages with": int(has.sum()),
                "CER with": cer(d[has]),
                "CER without": cer(d[~has]),
            }
        )
ops_table = pd.DataFrame(rows)
ops_table["difference"] = ops_table["CER with"] - ops_table["CER without"]
ops_table = ops_table.sort_values(["level", "difference"], ascending=[True, False])
with pd.option_context("display.precision", 3, "display.width", 200):
    display(ops_table.set_index(["level", "augmentation"]))

## how much of the error is accents, breathings and capitals?

In [ ]:
# --- How much of the reading error is accents, breathings and capitals? -----------
# Same reading on the TRUE lines (from the cache), scored three ways:
#   exact            : as in the benchmark (NFC)
#   no diacritics    : accents, breathings, iota subscripts removed on both sides
#   letters only     : no diacritics, lower case, punctuation removed
def no_diacritics(s):
    return unicodedata.normalize(
        "NFC",
        "".join(
            c
            for c in unicodedata.normalize("NFD", normalize_text(s))
            if unicodedata.category(c) != "Mn"
        ),
    )


def letters_only(s):
    return " ".join(strip_punct(no_diacritics(s)).lower().split())


rows = []
for model in [kraken, kraken_base]:
    for level in BENCH_LEVELS:
        n = Counter()
        for stem in pages:
            image, ann = load_page(level, stem)
            texts, _ = cached(
                model,
                level,
                stem,
                "read_oracle",
                lambda: model.read_lines(image, gt_line_input(ann)),  # noqa: B023
            )
            for line, pred in zip(gt_lines(ann), texts, strict=False):
                for name, f in [
                    ("exact", normalize_text),
                    ("no diacritics", no_diacritics),
                    ("letters only", letters_only),
                ]:
                    g, p = f(line["text_visible"]), f(pred)
                    n[name, "edits"] += edit_distance(g, p)
                    n[name, "chars"] += len(g)
                    n[name, "wedits"] += edit_distance(g.split(), p.split())
                    n[name, "words"] += len(g.split())
        row = {"model": model.name, "level": level}
        for name in ["exact", "no diacritics", "letters only"]:
            row[f"CER {name}"] = n[name, "edits"] / n[name, "chars"]
            row[f"WER {name}"] = n[name, "wedits"] / n[name, "words"]
        rows.append(row)
with pd.option_context("display.precision", 3, "display.width", 200):
    display(pd.DataFrame(rows).set_index(["model", "level"]))

In [ ]:
# --- Controlled test: one augmentation at a time, at fixed strengths ---------------
# Clean test page -> ONE augmentation at a given strength -> the model reads the TRUE
# lines (read_lines), so neither line finding nor other augmentations get mixed in.
# Same random draws for every strength of one augmentation (only the strength changes).
# Results are cached: an interrupted run resumes, a re-run is free.
import copy

SWEEP_MODELS = [kraken]  # add kraken_base to compare (doubles the time)
SWEEP_PAGES = benchmark_pages(
    8
)  # fixed subset of test pages; None-equivalent: benchmark_pages()
BG = 50  # "table" colour around a rotated / warped page


def _rotate(img, polys, rng, angle):
    h, w = img.shape[:2]
    H = np.vstack([cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0), [0, 0, 1]])
    return _warp_homography(img, polys, H, BG)


# name -> (strengths, geometric?, function(img, polys, rng, strength) -> (img, polys))
SWEEPS = {
    "tilt (degrees)": (
        [1, 2, 3, 5, 8, 12, 20],
        True,
        lambda img, P, rng, v: _rotate(img, P, rng, v),
    ),
    "perspective (corner shift)": (
        [0.02, 0.05, 0.08],
        True,
        lambda img, P, rng, v: perspective(
            img, P, rng, max_corner_shift=v, max_rotation=0, background=BG
        ),
    ),
    "page curl (amplitude)": (
        [0.004, 0.008, 0.012],
        True,
        lambda img, P, rng, v: page_curl(img, P, rng, max_amplitude=v),
    ),
    "noise (std)": (
        [2, 5, 10, 20],
        False,
        lambda img, P, rng, v: noise(img, P, rng, std_range=(v, v)),
    ),
    "blur (sigma, px)": (
        [0.5, 1, 1.5, 2, 3],
        False,
        lambda img, P, rng, v: (cv2.GaussianBlur(img, (0, 0), v), P),
    ),
    "motion blur (length, px)": (
        [3, 5, 9, 13],
        False,
        lambda img, P, rng, v: motion_blur(img, P, rng, length_range=(v, v + 1)),
    ),
    "low resolution (scale)": (
        [0.75, 0.5, 0.35, 0.25],
        False,
        lambda img, P, rng, v: low_resolution(img, P, rng, factor_range=(v, v)),
    ),
    "jpeg (quality)": (
        [80, 50, 30, 15],
        False,
        lambda img, P, rng, v: jpeg(img, P, rng, quality_range=(v, v + 1)),
    ),
    "shade (darkness)": (
        [0.2, 0.4, 0.6],
        False,
        lambda img, P, rng, v: shade(img, P, rng, darkness_range=(v, v)),
    ),
    "contrast (factor)": (
        [0.8, 0.6, 0.4],
        False,
        lambda img, P, rng, v: contrast_brightness(
            img, P, rng, contrast_range=(v, v), brightness_range=(0, 0)
        ),
    ),
}
STRONGER_IS_LOWER = {"low resolution (scale)", "jpeg (quality)", "contrast (factor)"}
CLEAN = {
    "low resolution (scale)": 1.0,
    "jpeg (quality)": 100,
    "contrast (factor)": 1.0,
}  # others: 0


def single_op_page(stem, op, v):
    """Clean page with ONE augmentation; ground truth moved with it if geometric."""
    image, ann = load_page("original", stem)
    _, geometric, fn = SWEEPS[op]
    rng = np.random.default_rng(
        [AUG_SEED, zlib.crc32(op.encode()), zlib.crc32(stem.encode())]
    )
    if not geometric:
        return fn(image, [], rng, v)[0], ann
    ann = copy.deepcopy(ann)
    L, W = ann["lines"], ann["words"]
    img, polys = fn(
        image, [densify(np.array(x["polygon"], np.float32)) for x in L + W], rng, v
    )
    h, w = img.shape[:2]
    for x, p in zip(L + W, polys, strict=False):
        p = finalize_shape(p, w, h, True)
        x["polygon"], x["bbox"] = p.tolist(), list(polygon_bbox(p.astype(np.float32)))
    for x, b in zip(
        W, ink_boxes(img, [np.array(x["polygon"]) for x in W]), strict=False
    ):  # before any blur
        x["ink_bbox"] = None if b is None else list(b)
    for line in L:
        lw = [x for x in W if x["line_id"] == line["line_id"]]
        line["words_bbox"] = box_union([x["bbox"] for x in lw])
        line["ink_bbox"] = box_union([x["ink_bbox"] for x in lw])
    return img, ann


def letters_counts(gt_texts, texts):
    n = Counter()
    for g, p in zip(
        gt_texts, list(texts) + [""] * (len(gt_texts) - len(texts)), strict=False
    ):
        g, p = letters_only(g), letters_only(p)
        n["edits"] += edit_distance(g, p)
        n["chars"] += len(g)
    return n


rows = []
steps = [
    (op, v)
    for op, (values, _, _) in SWEEPS.items()
    for v in [CLEAN.get(op, 0)] + values
]
for model in SWEEP_MODELS:
    for stem in tqdm(SWEEP_PAGES, desc=f"{model.name} sweep"):
        for op, v in steps:
            # the clean page (strength 0, or scale 1 / quality 100 / factor 1) is the
            # benchmark's "original" page: same cache entry, read once for all panels
            clean = v == CLEAN.get(op, 0)
            tag = (
                "original"
                if clean
                else f"sweep_{op.split(' (')[0].replace(' ', '_')}_{v}"
            )
            image, ann = (
                load_page("original", stem) if clean else single_op_page(stem, op, v)
            )
            G = gt_lines(ann)
            gt_texts = [line["text_visible"] for line in G]
            texts, words = cached(
                model,
                tag,
                stem,
                "read_oracle",
                lambda: model.read_lines(image, gt_line_input(ann)),  # noqa: B023
            )
            s = score_text(
                gt_texts, texts, [(i, i) for i in range(min(len(G), len(texts)))], {}
            )
            lc = letters_counts(gt_texts, texts)
            ws = score_words(ann, words)
            rows.append(
                {
                    "model": model.name,
                    "augmentation": op,
                    "strength": v,
                    "page": stem,
                    "char_edits": s["char_edits"],
                    "chars": s["chars"],
                    "word_edits": s["word_edits"],
                    "words": s["words"],
                    "letter_edits": lc["edits"],
                    "letters": lc["chars"],
                    "w_correct": ws["correct"],
                    "w_pred": ws["n_pred"],
                    "w_gt": ws["n_gt"],
                }
            )

sweep = pd.DataFrame(rows)
g = sweep.groupby(["model", "augmentation", "strength"], sort=False).sum(
    numeric_only=True
)
sweep_summary = pd.DataFrame(
    {
        "CER": g["char_edits"] / g["chars"],
        "CER letters only": g["letter_edits"] / g["letters"],
        "WER": g["word_edits"] / g["words"],
        "word F1 (box + text right)": 2 * g["w_correct"] / (g["w_pred"] + g["w_gt"]),
    }
)
sweep_summary.to_csv(AUG_DIR / "results" / "sweep_summary.csv")
with pd.option_context("display.precision", 3, "display.max_rows", 200):
    display(sweep_summary)

In [ ]:
# --- Plot: reading error against strength, one panel per augmentation -------------
SERIES_COLORS = ["#2a78d6", "#eb6834"]  # one colour per model, in SWEEP_MODELS order
ops = list(SWEEPS)
n_cols = 5
n_rows = int(np.ceil(len(ops) / n_cols))
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(3.4 * n_cols, 2.9 * n_rows), sharey=True
)
for ax in axes.flat:
    ax.set_visible(False)
y_max = max(
    1.0, 1.05 * sweep_summary[["CER", "CER letters only"]].max().max()
)  # CER can pass 1

for ax, op in zip(axes.flat, ops, strict=False):
    ax.set_visible(True)
    for model, color in zip(SWEEP_MODELS, SERIES_COLORS, strict=False):
        d = sweep_summary.loc[(model.name, op)].sort_index()
        ax.plot(
            d.index,
            d["CER"],
            color=color,
            lw=2,
            marker="o",
            ms=4,
            label=f"{model.name}: exact",
        )
        ax.plot(
            d.index,
            d["CER letters only"],
            color=color,
            lw=1.5,
            ls="--",
            marker="o",
            ms=3,
            label=f"{model.name}: letters only",
        )
    if op in STRONGER_IS_LOWER:
        ax.invert_xaxis()  # left = clean, right = strongest, like the other panels
    ax.set_title(op, fontsize=10, loc="left")
    ax.set_ylim(0, y_max)
    ax.grid(color="#e4e3df", lw=0.6)
    ax.tick_params(labelsize=8, colors="#555")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
for row in axes:
    row[0].set_ylabel("CER on true lines", fontsize=9)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, fontsize=9)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

We can see that the biggest problem for the models is the tilt of the image, so if we straighten the page, it probably permit to reduce the error of all models. This is free from training.